# Data Enrichment
1. Read the concatinated data
2. Handle Missing values
3. Create derived columns for further analyssi
4. Save the data in `data/JC`

## Loading Libraries

In [6]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

## Observe Missing Values

In [7]:
OUTPUT_DIR = "../data/JC/"

In [8]:
citibike_df = pd.read_csv(f"{OUTPUT_DIR}JC2025.csv", parse_dates=(["started_at", "ended_at"]))
print(citibike_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1002704 entries, 0 to 1002703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   ride_id             1002704 non-null  str           
 1   rideable_type       1002704 non-null  str           
 2   started_at          1002704 non-null  datetime64[us]
 3   ended_at            1002704 non-null  datetime64[us]
 4   start_station_name  1002701 non-null  str           
 5   start_station_id    1002701 non-null  str           
 6   end_station_name    999469 non-null   str           
 7   end_station_id      998307 non-null   str           
 8   start_lat           1002702 non-null  float64       
 9   start_lng           1002702 non-null  float64       
 10  end_lat             999260 non-null   float64       
 11  end_lng             999260 non-null   float64       
 12  member_casual       1002704 non-null  str           
dtypes: datetime64[us](2), f

In [9]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,4397,0.004385
10,end_lat,3444,0.003435
11,end_lng,3444,0.003435
6,end_station_name,3235,0.003226
4,start_station_name,3,0.000003
5,start_station_id,3,0.000003
8,start_lat,2,0.000002
9,start_lng,2,0.000002
0,ride_id,0,0.000000
1,rideable_type,0,0.000000


## Derived Columns
1. ride_duration_min
2. filter out anomalies
3. drop missing values
4. date
5. month
6. month_name
7. month_number
8. day_of_week
9. hour
10. season

### Ride duration

In [10]:
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

### Filter out anomalies

In [11]:
citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

### Drop missing values

In [12]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng", 
        "end_station_name",
        "start_station_name",
        "end_station_id"
    ]
)

### Time based variables

In [13]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [14]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [ ]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

<class 'pandas.DataFrame'>
Index: 998281 entries, 0 to 1002703
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ride_id                998281 non-null  str           
 1   rideable_type          998281 non-null  str           
 2   started_at             998281 non-null  datetime64[us]
 3   ended_at               998281 non-null  datetime64[us]
 4   start_station_name     998281 non-null  str           
 5   start_station_id       998281 non-null  str           
 6   end_station_name       998281 non-null  str           
 7   end_station_id         998281 non-null  str           
 8   start_lat              998281 non-null  float64       
 9   start_lng              998281 non-null  float64       
 10  end_lat                998281 non-null  float64       
 11  end_lng                998281 non-null  float64       
 12  member_casual          998281 non-null  str           
 13 

## Store the data

In [16]:
citibike_df.to_csv(f"{OUTPUT_DIR}/JC2025_Enriched.csv", index = False)